[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/05_flow_matching/05_flow_matching.ipynb)

# 05 · 流匹配 Flow Matching（用 numpy 从零）

目标：从零实现 **流匹配 / rectified flow**——直线概率路径、回归速度场、Euler ODE 采样，在 2D 八高斯分布上训练并 **沿速度场积分采样**；体会**少步采样**与**统一扩散**。

路线：直线路径 + 目标速度(**对拍解析**) → 速度场网络 + 回归损失(**梯度检验**) → 训练 → **Euler ODE 采样收敛** → 步数-质量权衡 → ✏️ 练习（速度目标、概率路径、Euler、与扩散对比）→ 📖 答案 → 🧪 真实(Heun 积分器)胶囊。

> 心智模型：**学一张「这里的点该往哪推」的速度场地图；生成 = 从噪声出发照地图一步步流到数据。确定性、可少步、统一扩散**。

## 1 · 直线概率路径与目标速度（对拍解析）

rectified flow：对一对 (噪声 x0, 数据 x1) 走直线 `x_t=(1-t)x0+t·x1`。
沿直线速度恒定且解析：`dx_t/dt = x1 - x0`。**这就是要回归的目标速度**。

验证：数值微分 `x_t` 对 `t` 的导数 == 解析目标 `x1-x0`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_8gauss(n, g, r=2.5, std=0.15):
    '''2D 八高斯：8 个簇均匀分布在半径 r 的圆上。'''
    k = g.integers(0, 8, n)
    ang = k * np.pi / 4
    centers = np.stack([r*np.cos(ang), r*np.sin(ang)], axis=1)
    return centers + std * g.standard_normal((n, 2))

def interpolate(x0, x1, t):
    '''直线路径上的点：x_t = (1-t)x0 + t·x1。t:(N,1)'''
    return (1 - t) * x0 + t * x1

def target_velocity(x0, x1):
    '''直线路径的目标速度（恒定）：x1 - x0'''
    return x1 - x0

x0 = rng.standard_normal((100, 2))
x1 = make_8gauss(100, rng)
# 数值微分 d x_t / d t 应等于 x1 - x0（与 t 无关）
t = np.full((100, 1), 0.4)
h = 1e-6
num_vel = (interpolate(x0, x1, t+h) - interpolate(x0, x1, t-h)) / (2*h)
ana_vel = target_velocity(x0, x1)
print('数值速度[0] =', np.round(num_vel[0], 4))
print('解析速度[0] =', np.round(ana_vel[0], 4))
assert np.allclose(num_vel, ana_vel, atol=1e-5), '直线速度应恒为 x1-x0'
# 端点检查：t=0 在噪声端，t=1 在数据端
assert np.allclose(interpolate(x0, x1, np.zeros((100,1))), x0)
assert np.allclose(interpolate(x0, x1, np.ones((100,1))), x1)
print('✅ 直线路径：t=0 噪声端、t=1 数据端，速度恒为 x1-x0（解析目标）')

## 2 · 速度场网络 v_θ(x,t) + 回归损失（梯度检验）

网络输入 `[x, t]`（2D 数据 + 1D 时刻 = 3 维），输出 2D 速度向量。
流匹配损失：`L = E‖v_θ(x_t,t) - (x1-x0)‖²`——回归目标速度。手写反向，数值梯度检验。

In [ ]:
def init_vnet(H=64, seed=0, s=0.3):
    g = np.random.default_rng(seed)
    return dict(W1=s*g.standard_normal((3,H)), b1=np.zeros(H),
                W2=s*g.standard_normal((H,H)), b2=np.zeros(H),
                W3=s*g.standard_normal((H,2)), b3=np.zeros(2))

def v_forward(x, t, P):
    '''x:(N,2), t:(N,1) -> 速度 (N,2)'''
    inp = np.concatenate([x, t], axis=1)
    h1 = np.tanh(inp @ P['W1'] + P['b1'])
    h2 = np.tanh(h1 @ P['W2'] + P['b2'])
    out = h2 @ P['W3'] + P['b3']
    return out, (inp, h1, h2)

def fm_loss_and_grads(xt, t, v_target, P):
    N = xt.shape[0]
    out, (inp, h1, h2) = v_forward(xt, t, P)
    loss = np.mean(np.sum((out - v_target)**2, axis=1))
    dout = (2.0 / N) * (out - v_target)
    g = {}
    g['W3'] = h2.T @ dout; g['b3'] = dout.sum(0)
    dh2 = (dout @ P['W3'].T) * (1 - h2**2)
    g['W2'] = h1.T @ dh2; g['b2'] = dh2.sum(0)
    dh1 = (dh2 @ P['W2'].T) * (1 - h1**2)
    g['W1'] = inp.T @ dh1; g['b1'] = dh1.sum(0)
    return loss, g

P = init_vnet()
xt = rng.standard_normal((32, 2)); tt = rng.random((32, 1)); vt = rng.standard_normal((32, 2))
loss, grads = fm_loss_and_grads(xt, tt, vt, P)
maxerr = 0.0
for key in ['W1', 'W2', 'W3']:
    W = P[key]; gn = np.zeros_like(W); it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index; o = W[i]
        W[i]=o+1e-6; fp=fm_loss_and_grads(xt,tt,vt,P)[0]
        W[i]=o-1e-6; fm=fm_loss_and_grads(xt,tt,vt,P)[0]; W[i]=o
        gn[i]=(fp-fm)/2e-6; it.iternext()
    maxerr = max(maxerr, np.max(np.abs(gn - grads[key])))
print('梯度检验 max|err| = %.2e' % maxerr)
assert maxerr < 1e-5, '速度场回归反向应与数值梯度一致'
print('✅ 速度场网络 + 回归损失就位，反向写对了')

## 3 · 训练速度场：回归 x1-x0

训练循环朴素如监督回归：抽 (噪声 x0, 数据 x1, 时刻 t)、算直线中点 x_t 和目标速度 x1-x0、MSE 下降。

**无 ODE 积分、无对抗、无 KL**——这就是流匹配「无模拟训练」的高效之处。

In [ ]:
def train_fm(epochs=4000, lr=0.01, H=64, bs=256, seed=0):
    P = init_vnet(H=H, seed=seed)
    g = np.random.default_rng(7); hist = []
    for ep in range(epochs):
        x0 = g.standard_normal((bs, 2))         # 先验噪声
        x1 = make_8gauss(bs, g)                 # 数据
        t = g.random((bs, 1))                   # [0,1] 时刻
        xt = (1 - t) * x0 + t * x1              # 直线中点
        v_target = x1 - x0                      # 目标速度（解析）
        loss, grads = fm_loss_and_grads(xt, t, v_target, P)
        for k in P: P[k] -= lr * grads[k]
        hist.append(loss)
    return P, hist

P_t, hist = train_fm(epochs=4000, lr=0.01)
early = np.mean(hist[:200]); late = np.mean(hist[-200:])
print('回归损失(平滑): %.3f -> %.3f' % (early, late))
assert late < early * 0.8, '训练应降低速度回归损失'
win = [np.mean(hist[i:i+200]) for i in range(0, len(hist)-200, 200)]
assert win[-1] < win[0], '损失应稳定下降（无对抗振荡）'
print('✅ 速度场训练完成，损失平滑下降 —— 流匹配训练稳如监督学习')

## 4 · Euler ODE 采样：沿速度场积分

从噪声 `x0~N(0,I)` 出发，Euler 法解 `dx/dt=v_θ`：`x_{t+Δt}=x_t+Δt·v_θ(x_t,t)`，走 N 步到 t=1。

采样后看生成是否覆盖**全部 8 个高斯簇**（不坍塌）、半径是否对（≈2.5）。

In [ ]:
def euler_sample(P, n, steps, seed=0):
    g = np.random.default_rng(seed)
    x = g.standard_normal((n, 2))               # x(0) ~ N(0,I)
    dt = 1.0 / steps
    for i in range(steps):
        t = np.full((n, 1), i * dt)
        v, _ = v_forward(x, t, P)
        x = x + dt * v                          # Euler 一步
    return x

def cluster_coverage(samples, r=2.5):
    '''每个样本归到最近的 8 个中心之一，返回(覆盖的簇数, 各簇计数)。'''
    ang = np.arange(8) * np.pi / 4
    C = np.stack([r*np.cos(ang), r*np.sin(ang)], axis=1)
    d = ((samples[:, None, :] - C[None, :, :])**2).sum(-1)
    lab = d.argmin(1)
    counts = np.bincount(lab, minlength=8)
    return int((counts > len(samples)*0.02).sum()), counts

fake = euler_sample(P_t, 3000, steps=50, seed=99)
nc, counts = cluster_coverage(fake)
radius = np.sqrt((fake**2).sum(1)).mean()
print('覆盖簇数 = %d / 8' % nc)
print('各簇样本数 =', counts)
print('生成点平均半径 = %.2f (目标≈2.5)' % radius)
assert nc == 8, '应覆盖全部 8 个高斯簇（流匹配不易坍塌）'
assert counts.min() > 3000 * 0.03, '各簇都应有相当样本（均衡覆盖）'
assert abs(radius - 2.5) < 0.4, '生成应落在正确半径上'
print('✅ 流匹配从噪声沿速度场流到 8 个簇，全覆盖、均衡、半径正确！')

## 5 · 步数-质量权衡：流匹配能少步采样

流匹配的杀手锏：**确定性 ODE + 直线路径 -> 很少步数就能出好样本**（对比扩散动辄几十上百步）。
看看从 4 步到 50 步，覆盖与半径如何变化——你会发现**少到几步也相当不错**。

In [ ]:
print(f"{'步数':>6} {'覆盖簇':>8} {'平均半径':>10} {'最少簇计数':>12}")
results = {}
for steps in [2, 4, 8, 20, 50]:
    fk = euler_sample(P_t, 3000, steps=steps, seed=99)
    nc, counts = cluster_coverage(fk)
    r = np.sqrt((fk**2).sum(1)).mean()
    results[steps] = (nc, r, counts.min())
    print(f'{steps:>6} {nc:>8} {r:>10.2f} {counts.min():>12}')
# 即便很少步(8)也应覆盖全部簇
assert results[8][0] == 8, '8 步就应覆盖全部 8 簇（流匹配少步优势）'
# 步数越多半径越接近目标（积分越精确）
assert abs(results[50][1] - 2.5) <= abs(results[2][1] - 2.5) + 0.05, '步数越多通常越精确'
print('✅ 流匹配少步采样威力：8 步就覆盖全部簇，远少于扩散的几十步！')

## 6 · 流匹配 ⇔ 扩散：速度 vs 噪声（统一）

流匹配预测**速度** `x1-x0`，扩散预测**噪声** `ε`。两者都是「在中间点回归一个指向数据的向量」。
这里用一个简单的 1D 类比展示：对高斯先验，**速度目标与噪声目标可以互相换算**（统一的核心）。

In [ ]:
# 1D 类比：x_t = (1-t)x0 + t·x1，速度 = x1 - x0
# 若把 x0 视作'噪声'(~N(0,1))、x1 视作'数据'，则：
#   已知 x_t 和 t，速度 v=x1-x0 与 '噪声' x0 之间是线性关系（可互解）
g = np.random.default_rng(1)
x0 = g.standard_normal((1000, 1))      # '噪声'
x1 = (g.integers(0,2,1000)*4 - 2).reshape(-1,1) + 0.2*g.standard_normal((1000,1))  # '数据'(双峰)
t = g.random((1000, 1))
xt = (1-t)*x0 + t*x1
v = x1 - x0                            # 速度目标
# 从 (xt, t, v) 反解 x0：x0 = xt - t·(x1-x0)... 用 x1 = xt + (1-t)*v 推出
# 已知 xt=(1-t)x0+t x1 且 v=x1-x0  =>  x1 = xt + (1-t)v,  x0 = xt - t·v
x0_rec = xt - t * v
x1_rec = xt + (1 - t) * v
assert np.allclose(x0_rec, x0, atol=1e-9), '由(x_t,v)可精确反解噪声 x0'
assert np.allclose(x1_rec, x1, atol=1e-9), '由(x_t,v)可精确反解数据 x1'
print('✅ 速度 v=x1-x0 与噪声 x0 在中间点 x_t 上线性可互解')
print('   => 预测速度(流匹配) ⇔ 预测噪声(扩散) 是一体两面，流匹配统一了扩散')

---
## ✏️ 练习 1：目标速度场

实现 `velocity_target(x0, x1, path='linear')`：对直线路径返回目标速度 `x1-x0`。（直线路径的速度与 t 无关，恒定。）

In [ ]:
def velocity_target(x0, x1, path='linear'):
    # TODO: 直线路径目标速度 = x1 - x0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
a = rng.standard_normal((20, 2)); b = rng.standard_normal((20, 2))
v = velocity_target(a, b)
assert np.allclose(v, b - a)
# 速度指向数据方向：x_t 沿 v 走 dt，应更接近 x1
t = 0.3; xt = (1-t)*a + t*b
xt_next = xt + 0.01 * v
assert np.all(np.sum((xt_next-b)**2,1) < np.sum((xt-b)**2,1)), '沿速度走应更接近数据'
print('✅ 练习 1 通过：目标速度 = x1-x0，指向数据方向')

## ✏️ 练习 2：概率路径上的点

实现 `sample_path_point(x0, x1, t)`：返回直线路径上时刻 `t` 的点 `(1-t)x0+t·x1`。并验证它在 `t=0` 是噪声、`t=1` 是数据、`t=0.5` 是中点。

In [ ]:
def sample_path_point(x0, x1, t):
    # TODO: (1-t)*x0 + t*x1
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
a = rng.standard_normal((10, 2)); b = rng.standard_normal((10, 2))
assert np.allclose(sample_path_point(a, b, np.zeros((10,1))), a)
assert np.allclose(sample_path_point(a, b, np.ones((10,1))), b)
assert np.allclose(sample_path_point(a, b, np.full((10,1),0.5)), 0.5*(a+b))
print('✅ 练习 2 通过：直线路径点正确（t=0噪声、t=1数据、t=0.5中点）')

## ✏️ 练习 3：一步 Euler 积分

实现 `euler_step(x, t, dt, v_fn, P)`：执行一步 Euler `x + dt·v_fn(x,t,P)[0]`（`v_fn` 返回 `(速度,缓存)`）。返回新位置。

In [ ]:
def euler_step(x, t, dt, v_fn, P):
    # TODO: v = v_fn(x, t, P)[0]; 返回 x + dt*v
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x = rng.standard_normal((20, 2)); t = np.full((20,1), 0.0)
x_new = euler_step(x, t, 0.1, v_forward, P_t)
v0 = v_forward(x, t, P_t)[0]
assert np.allclose(x_new, x + 0.1 * v0), '应为 x + dt*v'
# 多步 Euler 从噪声到数据，半径应增大(从~1.4 -> ~2.5)
xx = rng.standard_normal((1000, 2)); r0 = np.sqrt((xx**2).sum(1)).mean()
for i in range(20):
    xx = euler_step(xx, np.full((1000,1), i/20), 1/20, v_forward, P_t)
r1 = np.sqrt((xx**2).sum(1)).mean()
assert r1 > r0, 'Euler 积分应把点从噪声(半径小)推到数据环(半径大)'
print('✅ 练习 3 通过：Euler 一步正确，多步把噪声流到数据环(半径 %.2f->%.2f)' % (r0, r1))

## ✏️ 练习 4：流匹配 vs 扩散——换算速度与噪声

对直线路径 `x_t=(1-t)x0+t·x1`、速度 `v=x1-x0`：实现 `recover_endpoints(xt, v, t)`，由 `(x_t, v, t)` 反解 `(x0, x1)`。提示：`x1=xt+(1-t)v`，`x0=xt-t·v`。

In [ ]:
def recover_endpoints(xt, v, t):
    # TODO: x0 = xt - t*v ; x1 = xt + (1-t)*v ; 返回 (x0, x1)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
a = rng.standard_normal((30, 2)); b = rng.standard_normal((30, 2)); t = rng.random((30,1))
xt = (1-t)*a + t*b; v = b - a
x0_rec, x1_rec = recover_endpoints(xt, v, t)
assert np.allclose(x0_rec, a, atol=1e-9) and np.allclose(x1_rec, b, atol=1e-9)
print('✅ 练习 4 通过：速度与端点可精确互解 —— 这是流匹配统一扩散的代数基础')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def velocity_target(x0, x1, path='linear'):
    return x1 - x0

In [ ]:
# 练习 2 参考答案
def sample_path_point(x0, x1, t):
    return (1 - t) * x0 + t * x1

In [ ]:
# 练习 3 参考答案
def euler_step(x, t, dt, v_fn, P):
    v = v_fn(x, t, P)[0]
    return x + dt * v

In [ ]:
# 练习 4 参考答案
def recover_endpoints(xt, v, t):
    x0 = xt - t * v
    x1 = xt + (1 - t) * v
    return x0, x1

---
## 🧪 真实技巧胶囊：Heun 二阶积分器（更准、可更少步）

Euler 是一阶，误差 `O(Δt)`。**Heun 法**（改进欧拉，二阶）先用 Euler 预测、再用两端速度的平均校正，误差 `O(Δt²)`——同样步数下更准，常用于少步采样。这里从零实现并对比。**纯本地计算**。

In [ ]:
def heun_sample(P, n, steps, seed=0):
    '''Heun(改进欧拉)：x* = x + dt·v(x,t); x' = x + dt/2·(v(x,t)+v(x*,t+dt))'''
    g = np.random.default_rng(seed)
    x = g.standard_normal((n, 2)); dt = 1.0 / steps
    for i in range(steps):
        t = np.full((n, 1), i * dt)
        v1, _ = v_forward(x, t, P)
        x_pred = x + dt * v1                      # Euler 预测
        v2, _ = v_forward(x_pred, np.full((n,1),(i+1)*dt), P)
        x = x + dt * 0.5 * (v1 + v2)             # 两端速度平均校正
    return x

# 对比少步(4步)下 Euler vs Heun 的半径精度
fake_euler = euler_sample(P_t, 3000, steps=4, seed=99)
fake_heun  = heun_sample(P_t, 3000, steps=4, seed=99)
r_euler = np.sqrt((fake_euler**2).sum(1)).mean()
r_heun  = np.sqrt((fake_heun**2).sum(1)).mean()
print('4 步采样平均半径(目标2.5): Euler=%.3f  Heun=%.3f' % (r_euler, r_heun))
print('✅ 数据就绪：Heun 在少步下通常更接近目标半径')

**🧪 胶囊练习**：实现 `radius_error(samples, target_r=2.5)`：返回生成点平均半径与目标半径的绝对差。用它验证少步时 Heun 比 Euler 更准（误差更小或相当）。

In [ ]:
def radius_error(samples, target_r=2.5):
    # TODO: |mean(sqrt(sum(samples^2,axis=1))) - target_r|
    raise NotImplementedError

In [ ]:
# 自测
err_euler = radius_error(fake_euler)
err_heun = radius_error(fake_heun)
print('半径误差: Euler=%.3f  Heun=%.3f' % (err_euler, err_heun))
assert err_heun <= err_euler + 0.05, '少步时 Heun 应不差于 Euler(通常更准)'
# 多步时两者都该很准
assert radius_error(heun_sample(P_t, 2000, 30, 1)) < 0.4
print('✅ 胶囊练习通过：高阶积分器(Heun)在少步采样下更准 —— 通往更少步生成')

In [ ]:
# 📖 胶囊参考答案
def radius_error(samples, target_r=2.5):
    return abs(np.sqrt((samples**2).sum(1)).mean() - target_r)

### 小结
- 流匹配 = 学一个把噪声分布**搬运**到数据分布的**速度场** v_θ(x,t)；生成 = 沿它积分 ODE。
- **直线路径**(rectified flow)目标速度恒为 `x1-x0`（解析）；训练是朴素回归——**无 ODE 积分、无对抗、无 KL**。
- **Euler ODE 采样**确定性、可**少步**（8 步就覆盖全部簇）；高阶积分器(Heun)更准。
- **预测速度 ⇔ 预测噪声**：流匹配统一了扩散（扩散的概率流 ODE 是流匹配的一个特例）。
- 流匹配是扩散的「连续/统一/高效」版，是现代文生图(SD3/Flux)的主流框架。

🎉 **你已走完基础生成模型课的五族模型！** 下一步：**前沿扩散课 C28**（DiT、score-based SDE、概率流 ODE、引导与蒸馏、一步生成）——本课打下的地基会让那些前沿一点就通。